In [6]:
import anndata as ad
import scanpy as sc

In [7]:
adata = ad.read_h5ad("data/hnc_myeloid_2021.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 26444 × 23630
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters', 'RNA_snn_res.0.5', 'RNA_snn_res.0.6', 'RNA_snn_res.0.7', 'RNA_snn_res.0.8', 'RNA_snn_res.0.9', 'RNA_snn_res.1', 'RNA_snn_res.1.1', 'RNA_snn_res.1.2', 'RNA_snn_res.1.3', 'RNA_snn_res.1.4', 'RNA_snn_res.1.5', 'RNA_snn_res.1.6', 'RNA_snn_res.1.7', 'RNA_snn_res.1.8', 'RNA_snn_res.1.9', 'RNA_snn_res.2', 'tissue', 'is_HD', 'global.cluster', 'global.cluster2', 'hpv_status', 'RNA_snn_res.2.1', 'RNA_snn_res.2.2', 'RNA_snn_res.2.3', 'RNA_snn_res.2.4', 'RNA_snn_res.2.5', 'RNA_snn_res.2.6', 'RNA_snn_res.2.7', 'RNA_snn_res.2.8', 'RNA_snn_res.2.9', 'RNA_snn_res.3', 'tissue_hpv', 'global.cluster3', 'global.cluster4'
    obsm: 'X_harmony', 'X_pca', 'X_umap'


In [8]:
print(adata.X[:5, :5])              # peek at the matrix (.toarray() first if sparse)
print(type(adata.X))                # csr_matrix vs ndarray
print(adata.X.min(), adata.X.max()) # raw counts vs log-normalized tell
print(adata.obs.head())             # cell metadata
print(adata.var.head())             # gene metadata
print(adata.layers.keys())          # often where raw counts hide
print(adata.obs.columns.tolist())   # all the per-cell columns

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 0 stored elements and shape (5, 5)>
<class 'scipy.sparse._csr.csr_matrix'>
0 17185
                                          orig.ident  nCount_RNA  \
GSM4138110_HNSCC_1_PBMC_AAACCTGAGGAGCGTT  GSM4138110        1077   
GSM4138110_HNSCC_1_PBMC_AAACCTGCAGATGAGC  GSM4138110        2351   
GSM4138110_HNSCC_1_PBMC_AACGTTGAGACTGGGT  GSM4138110         946   
GSM4138110_HNSCC_1_PBMC_AACGTTGGTTACGACT  GSM4138110        1088   
GSM4138110_HNSCC_1_PBMC_AACGTTGTCTAAGCCA  GSM4138110        1217   

                                          nFeature_RNA  seurat_clusters  \
GSM4138110_HNSCC_1_PBMC_AAACCTGAGGAGCGTT           550                3   
GSM4138110_HNSCC_1_PBMC_AAACCTGCAGATGAGC           992                4   
GSM4138110_HNSCC_1_PBMC_AACGTTGAGACTGGGT           455                3   
GSM4138110_HNSCC_1_PBMC_AACGTTGGTTACGACT           556                3   
GSM4138110_HNSCC_1_PBMC_AACGTTGTCTAAGCCA           618                4  

In [10]:
adata.layers["counts"] = adata.X.copy()

# global 5000 HVGs (seurat_v3 expects raw counts)
sc.pp.highly_variable_genes(adata, n_top_genes=5000, flavor="seurat_v3", layer="counts")
global_hvg = set(adata.var_names[adata.var.highly_variable])

# normalize+log for DE
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# find which column holds the DC subsets
for c in ["global.cluster4"]:
    print(c, adata.obs[c].unique())


/opt/anaconda3/lib/python3.13/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


global.cluster4 ['Mono_CD14', 'Mac_IL1Bint', 'cDC2_CD1C', 'Mac_CXCL9', 'Mono_TIL', ..., 'cDC1_CLEC9A', 'DC_pDC', 'Mono_CD14_ID1', 'Mac_SPP1', 'Mast']
Length: 17
Categories (17, object): ['DC_pDC', 'Mac_CXCL9', 'Mac_IL1B', 'Mac_IL1Bint', ..., 'cDC1_CLEC9A', 'cDC2_CD1C', 'cDC2_CD33', 'mregDC_LAMP3']


In [11]:
col = "global.cluster4"          # <- set to the one with DC labels
dc_labels = ["cDC1_CLEC9A", "cDC2_CD1C", "cDC2_CD33", "mregDC_LAMP3"]   # <- your actual names

dc = adata[adata.obs[col].isin(dc_labels)].copy()

# DE *among* the three subsets, not vs all cells
sc.tl.rank_genes_groups(dc, groupby=col, method="wilcoxon")
df = sc.get.rank_genes_groups_df(dc, group=None)
sig = df[(df.pvals_adj < 0.05) & (df.logfoldchanges.abs() > 0.5)]
top_de = set(sig.groupby("group").head(100)["names"])

print(len(top_de), "DE genes")
print(len(top_de & global_hvg), "in global 5000 HVGs")
print(sorted(top_de - global_hvg)[:40])   # the ones it missed

382 DE genes
189 in global 5000 HVGs
['ABI3', 'ACTN1', 'ACTR3', 'ALOX5', 'ANXA6', 'APH1A', 'APOL3', 'ARF4', 'ARHGAP18', 'ARL6IP5', 'ASAP1', 'ATG3', 'ATP1A1', 'ATP5D', 'ATP5G2', 'BATF3', 'BLVRB', 'BZW1', 'C14orf2', 'C1QBP', 'C1orf21', 'C20orf27', 'CADM1', 'CAMK2D', 'CASP1', 'CCND1', 'CD274', 'CD33', 'CD40', 'CD53', 'CD58', 'CDK2AP2', 'CERS6', 'CFLAR', 'CIB1', 'CIRBP', 'CLNK', 'CMTM6', 'CNPY3', 'COMMD6']


/var/folders/bw/grhyggnj06ncywf0093kz3ch0000gn/T/ipykernel_31546/879311505.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top_de = set(sig.groupby("group").head(100)["names"])
